# Bootstrap & Preflight — 09-Champion_AutoWire.ipynb

**What this does**
- Verifies expected upstream notebooks have been executed and essential artefact folders exist.
- Prints clear guidance to run prerequisites if required files/folders are missing.

**Upstream prerequisites (recommended order)**
- `01-Setup_Preflight.ipynb`
- `02-Feature_Engineering.ipynb`
- `03-Feature_Selection.ipynb`
- `04-Baseline_and_BO.ipynb`
- `06-Ensembles.ipynb`
- `07-CNN.ipynb`

**Checks performed**
- Confirms `DATA_PATH` exists (from Section 0.1).
- Ensures `staging/` and `out/` directories exist when required downstream.
- Provides actionable instructions when a check fails.


In [ ]:
# ======================================================
# Bootstrap & Preflight — 09-Champion_AutoWire.ipynb
#   • Validates prerequisites and artefact folders
#   • Prints guidance if prerequisites are missing
# ======================================================
print(">>> Bootstrap & Preflight — 09-Champion_AutoWire.ipynb")
required = ['01-Setup_Preflight.ipynb', '02-Feature_Engineering.ipynb', '03-Feature_Selection.ipynb', '04-Baseline_and_BO.ipynb', '06-Ensembles.ipynb', '07-CNN.ipynb']
print("[bootstrap] Recommended upstream notebooks:", required)

# Check DATA_PATH existence if declared
if 'DATA_PATH' in globals():
    from pathlib import Path as _P
    dp = _P(DATA_PATH)
    if not dp.exists():
        print(f"[bootstrap][warn] DATA_PATH not found: {dp} — please verify in 01-Setup_Preflight (Section 0.1).")

# Check staging/out directories
from pathlib import Path as _P
if 'STAGE_ROOT' in globals():
    sr = _P(STAGE_ROOT); 
    if not sr.exists():
        print(f"[bootstrap][warn] STAGE_ROOT does not exist: {sr}. Run 01-Setup_Preflight end-to-end first.")
if 'OUT_ROOT' in globals():
    oroot = _P(OUT_ROOT);
    if not oroot.exists():
        print(f"[bootstrap][warn] OUT_ROOT does not exist: {oroot}. It will be created as needed, but prior steps may be required.")

# Feature artefacts helpful for downstream
from pathlib import Path as _P
feat_dir = _P('staging') / 'feat'
if not feat_dir.exists():
    print("[bootstrap][hint] 'staging/feat' not found — this notebook can generate it (Sections 3.3/3.4), or run 03-Feature_Selection first.")
else:
    mi_file = feat_dir / 'mi_series.csv'
    if not mi_file.exists():
        print("[bootstrap][hint] MI series not found at 'staging/feat/mi_series.csv' — run Section 3.3 to generate.")

print("[bootstrap] Preflight checks complete. Proceed with this notebook if no critical warnings above.")


## Section 9 — Section

In [ ]:
# =====================================================
# Section 9 — Section
# =====================================================
print(">>> Section 9: start")
reports_dir = Path(OUT_ROOT) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
if champion_valid:
    with (reports_dir / 'champion.json').open('w') as f:
        json.dump(champion_obj, f, indent=2)
    print('[ok] champion.json saved →', reports_dir / 'champion.json')
else:
    precheck_missing = tuple(globals().get('PRECHECK_MISSING', ()))
    required_sections = ['Section 4 — Train LightGBM (baseline or BO)', 'Section 5.* — HPO variants (if used in selection)', 'Section 6.1 — Baseline Model & Metrics (if metrics aggregation happens there)']
    payload_needed = 'payload_seq' in precheck_missing
    if payload_needed:
        required_sections.append('Section 2.1 — Payload Sequence Preprocessing')
    pending = {'status': 'pending', 'reason': 'No model scores available for champion selection.', 'detected_missing': list(precheck_missing), 'required_next_steps': required_sections, 'notes': ['Upstream warnings indicated metrics computed with y_true=None.', 'Ensure validation splits and y labels are available before scoring.'], 'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    with (reports_dir / 'champion.json').open('w') as f:
        json.dump(pending, f, indent=2)
    CHAMPION_AVAILABLE = False
    CHAMPION_STATUS = 'pending'
    print('[skip] champion pending — generated reports/champion.json with next steps')

>>> Section 9: champion selection (resilient)
[skip] champion pending — generated reports/champion.json with next steps


## Section 9.1 — Section

## Section 9.2 — Section

## Section 9.3 — Section

In [ ]:
# =====================================================
# Section 9.3 — Section
# =====================================================
print(">>> Section 9.3: start")
reports_dir = Path(OUT_ROOT) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)
X_train_t = globals().get('X_train_t', None)
y_train = globals().get('y_train', None)

def _safe_1d_y(y):
    if y is None:
        return np.array([], dtype=int)
    arr = np.asarray(getattr(y, 'values', y))
    arr = np.atleast_1d(arr)
    if arr.ndim > 1:
        arr = arr.ravel()
    return arr
y_vec = _safe_1d_y(y_train)

def _can_run_cv(X, y):
    try:
        if X is None or y is None:
            return (False, 'features_or_labels_missing')
        if not hasattr(X, 'shape') or X.shape[0] == 0:
            return (False, 'no_samples')
        if y.size == 0:
            return (False, 'labels_empty')
        if np.unique(y).size < 2:
            return (False, 'need_at_least_two_classes')
        if X.shape[0] != y.size:
            return (False, f'length_mismatch_X{X.shape[0]}_y{y.size}')
        return (True, 'ok')
    except Exception as e:
        return (False, f'validation_error: {e}')
ok_run, why_not = _can_run_cv(X_train_t, y_vec)
models = {'lgbm_baseline': globals().get('baseline_model'), 'xgboost': globals().get('xgb_model'), 'manual_hpo': globals().get('manual_hpo_model')}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=globals().get('RANDOM_STATE', 42))
sc_ap = make_scorer(average_precision_score, needs_proba=True)
summary = {}
reports_written = 0
for name, model in models.items():
    report_path = reports_dir / f'cv_{name}.json'
    rep = {'model': name, 'status': None, 'reason': None, 'scores': None, 'mean_ap': None, 'n_splits': 3}
    if model is None:
        rep.update({'status': 'pending', 'reason': 'model_missing'})
    elif not ok_run:
        rep.update({'status': 'pending', 'reason': why_not})
    else:
        try:
            scores = cross_val_score(model, X_train_t, y_vec, cv=cv, scoring=sc_ap, error_score='raise')
            rep.update({'status': 'ok', 'reason': '', 'scores': [float(s) for s in scores], 'mean_ap': float(np.mean(scores))})
        except Exception as e:
            rep.update({'status': 'error', 'reason': str(e)})
    report_path.write_text(json.dumps(rep, indent=2))
    summary[name] = rep
    reports_written += 1
ensemble_rep = {'model': 'ensemble_soft', 'status': 'pending', 'reason': 'no_base_cv_ok', 'k': 1}
if any((r.get('status') == 'ok' for r in summary.values())):
    ok_items = [(k, v['mean_ap']) for k, v in summary.items() if v.get('status') == 'ok']
    best_name, best_ap = sorted(ok_items, key=lambda x: x[1], reverse=True)[0]
    ensemble_rep.update({'status': 'ok', 'reason': f'proxy_best={best_name}', 'best_base': best_name, 'best_ap': float(best_ap)})
(reports_dir / 'cv_ensemble_soft.json').write_text(json.dumps(ensemble_rep, indent=2))
summary['ensemble_soft'] = ensemble_rep
(reports_dir / 'cv_summary.json').write_text(json.dumps(summary, indent=2))
CV_AVAILABLE = any((v.get('status') == 'ok' for v in summary.values()))
print(f'[ok] wrote {reports_written + 2} CV report files → {reports_dir} | any_ok={CV_AVAILABLE}')

>>> Section 9.3: Cross-Validation Stability Testing (resilient)
[ok] wrote 5 CV report files → out/reports | any_ok=False


/opt/miniconda3/envs/py312/lib/python3.12/site-packages/sklearn/metrics/_scorer.py:548: FutureWarning: The `needs_threshold` and `needs_proba` parameter are deprecated in version 1.4 and will be removed in 1.6. You can either let `response_method` be `None` or set it to `predict` to preserve the same behaviour.
  warnings.warn(


## Section 9.6 — Section

In [ ]:
# =====================================================
# Section 9.6 — Section
# =====================================================
print(">>> Section 9.6: start")
reports.mkdir(parents=True, exist_ok=True)
models_dir = Path(OUT_ROOT) / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
stage = Path(STAGE_ROOT)

def _load_champion_path():
    p = models_dir / 'champion_deploy.pkl'
    if p.exists():
        return p
    cand = {}
    for tag in ['lgbm_baseline', 'xgboost', 'lgbm_optuna', 'manual_grid', 'manual_random', 'bayes_searchcv', 'ensemble_soft']:
        f = reports / f'pr_{tag}.json'
        if f.exists():
            try:
                d = json.loads(f.read_text())
                cand[tag] = float(d.get('ap', -1.0))
            except Exception:
                pass
    if not cand:
        raise RuntimeError('No champion found. Run Sections 9.3/9.4 first.')
    best_tag = max(cand, key=cand.get)
    mapping = {'lgbm_baseline': models_dir / 'lgbm_baseline.pkl', 'xgboost': models_dir / 'xgb_baseline.pkl', 'lgbm_optuna': models_dir / 'lgbm_optuna.pkl', 'manual_grid': models_dir / 'lgbm_grid.pkl', 'manual_random': models_dir / 'lgbm_random.pkl', 'bayes_searchcv': models_dir / 'lgbm_bayessearch.pkl', 'ensemble_soft': models_dir / 'ensemble_soft.pkl'}
    return mapping[best_tag]

def _load_preprocessor():
    for p in [stage / 'split_preproc' / 'preprocessor.pkl', stage / 'preprocessor.pkl']:
        if p.exists():
            try:
                return joblib.load(p)
            except Exception:
                pass
    return None

def predict_proba_realtime(df_raw: pd.DataFrame):
    """
    Real-time scoring entrypoint.
    - If a preprocessor has been staged, use it to transform df_raw.
    - Else, assume df_raw is already in transformed space aligned to training.
    Returns: np.ndarray of probabilities.
    """
    model_path = _load_champion_path()
    model = joblib.load(model_path)
    preproc = _load_preprocessor()
    if preproc is not None:
        X_new = preproc.transform(df_raw)
    else:
        X_new = df_raw
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_new)[:, 1]
    else:
        proba = model.predict(X_new)
        proba = np.asarray(proba).ravel()
    return proba
print('[ok] Inference utilities ready: call predict_proba_realtime(df_raw).')

>>> Section 9.6: Deployment & Real‑time Inference
[ok] Inference utilities ready: call predict_proba_realtime(df_raw).


## Section 9.7 — Section

In [ ]:
# =====================================================
# Section 9.7 — Section
# =====================================================
print(">>> Section 9.7: start")

def _load_transformed_reference():
    for p in [stage / 'split_preproc' / 'X_train_t.pkl', stage / 'X_train_t.pkl']:
        if p.exists():
            return joblib.load(p)
    raise FileNotFoundError('X_train_t not found in staging; run Sections 0.4/0.5.')

def _to_dense(arr):
    try:
        return arr.toarray()
    except Exception:
        return np.asarray(arr)

def psi(expected: np.ndarray, actual: np.ndarray, bins: int=20) -> float:
    eps = 1e-12
    q = np.linspace(0, 100, bins + 1)
    cuts = np.unique(np.percentile(expected, q))
    if cuts.size < 3:
        return 0.0
    exp_hist, _ = np.histogram(expected, bins=cuts)
    act_hist, _ = np.histogram(actual, bins=cuts)
    exp_pct = (exp_hist / max(exp_hist.sum(), 1)).astype(float) + eps
    act_pct = (act_hist / max(act_hist.sum(), 1)).astype(float) + eps
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def drift_report_from_batch(df_raw: pd.DataFrame, max_features: int=100):
    preproc = None
    for p in [stage / 'split_preproc' / 'preprocessor.pkl', stage / 'preprocessor.pkl']:
        if p.exists():
            try:
                preproc = joblib.load(p)
                break
            except Exception:
                pass
    X_ref = _to_dense(_load_transformed_reference())
    if preproc is not None:
        X_new = _to_dense(preproc.transform(df_raw))
    else:
        X_new = _to_dense(df_raw)
    F = min(X_ref.shape[1], X_new.shape[1], max_features)
    scores = []
    for j in range(F):
        s = psi(X_ref[:, j], X_new[:, j])
        scores.append(float(s))
    overall = float(np.nanmean(scores))
    ts = _dt.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
    out = {'timestamp': ts, 'psi_mean': overall, 'psi_per_feature': scores, 'features_evaluated': int(F)}
    out_path = reports / f'drift_report_{ts}.json'
    Path(out_path).write_text(json.dumps(out, indent=2))
    print(f'[ok] drift report → {out_path} (psi_mean={overall:.4f})')
    return (out_path, out)
print('[ok] Drift utilities ready: call drift_report_from_batch(df_raw).')

>>> Section 9.7: Data Drift Monitoring (PSI)
[ok] Drift utilities ready: call drift_report_from_batch(df_raw).


## Section 9.8 — Section

In [ ]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")
reports_dir = Path(OUT_ROOT) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

def _safe_load_json(p: Path):
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None
rows = []
for cand in ['lgbm_report.json', 'lightgbm_report.json', 'champion.json']:
    rep = _safe_load_json(reports_dir / cand)
    if rep:
        ap = rep.get('ap') or rep.get('mean_ap') or rep.get('metrics', {}).get('ap')
        row = {'model': rep.get('model', 'LightGBM (report)'), 'status': rep.get('status', 'ok' if ap is not None else 'pending'), 'ap': float(ap) if ap is not None else None, 'f1_at_tau': rep.get('f1_at_tau'), 'tau_star': rep.get('tau_star'), 'precision': rep.get('precision'), 'recall': rep.get('recall'), 'cm_tn_fp_fn_tp': rep.get('cm_tn_fp_fn_tp'), 'notes': rep.get('reason') or rep.get('notes', '')}
        rows.append(row)
        break
cnn_candidates = ['cnn_report.json', 'cnn_metrics.json', 'cnn_eval.json']
cnn_rep = None
for cand in cnn_candidates:
    tmp = _safe_load_json(reports_dir / cand)
    if tmp:
        cnn_rep = tmp
        break
if cnn_rep:
    ap = cnn_rep.get('ap') or cnn_rep.get('mean_ap') or cnn_rep.get('metrics', {}).get('ap')
    rows.append({'model': cnn_rep.get('model', 'CNN'), 'status': cnn_rep.get('status', 'ok' if ap is not None else 'pending'), 'ap': float(ap) if ap is not None else None, 'f1_at_tau': cnn_rep.get('f1_at_tau'), 'tau_star': cnn_rep.get('tau_star'), 'precision': cnn_rep.get('precision'), 'recall': cnn_rep.get('recall'), 'cm_tn_fp_fn_tp': cnn_rep.get('cm_tn_fp_fn_tp'), 'notes': cnn_rep.get('reason') or cnn_rep.get('notes', '')})
else:
    note = 'CNN metrics unavailable — run Section 2.1 (payload sequence) and Section 7 (CNN training).'
    rows.append({'model': 'CNN', 'status': 'pending', 'ap': None, 'f1_at_tau': None, 'tau_star': None, 'precision': None, 'recall': None, 'cm_tn_fp_fn_tp': None, 'notes': note})
df = pd.DataFrame(rows, columns=['model', 'status', 'ap', 'f1_at_tau', 'tau_star', 'precision', 'recall', 'cm_tn_fp_fn_tp', 'notes'])
display_df = df.copy()
metric_cols = ['ap', 'f1_at_tau', 'tau_star', 'precision', 'recall']
for c in metric_cols:
    mask = (display_df['status'] != 'ok') | display_df[c].isna()
    display_df.loc[mask, c] = '–'
    with np.errstate(all='ignore'):
        display_df.loc[~mask, c] = pd.to_numeric(display_df.loc[~mask, c], errors='coerce').round(6)
df.to_json(reports_dir / 'final_table_raw.json', orient='records', indent=2)
display_df.to_csv(reports_dir / 'final_table.csv', index=False)
display(display_df)
print(f'[ok] final reports written → {reports_dir} (final_table.csv, final_table_raw.json)')

>>> Section 9.8: compile final reports (robust CNN handling)


,model,status,ap,f1_at_tau,tau_star,precision,recall,cm_tn_fp_fn_tp,notes
0,LightGBM (report),pending,–,–,–,–,–,None,No model scores available for champion selection.
1,CNN,pending,–,–,–,–,–,None,CNN metrics unavailable — run Section 2.1 (pay...


[ok] final reports written → out/reports (final_table.csv, final_table_raw.json)


In [ ]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")

In [ ]:
# =====================================================
# Section 9.8 — Section
# =====================================================
print(">>> Section 9.8: start")

## Section 9.8 — Champion Summary

**What this does**  
- Compiles a concise summary of the selected champion and key metrics found in `out/metrics/`.  
- Prints a single table (winner, metric, model file, sources).

In [ ]:
# =====================================================
# Section 9.8 — Champion Summary
# =====================================================
print(">>> Section 9.8 — Champion Summary: start")

from pathlib import Path
import json

OUT_ROOT = globals().get("OUT_ROOT", "out")
out_dir = Path(OUT_ROOT)
metrics_dir = out_dir / "metrics"
reports_dir = out_dir / "reports"

best_tree = None
cnn_meta = None
try:
    p = metrics_dir / "best_tree.json"
    if p.exists():
        best_tree = json.loads(p.read_text())
except Exception:
    pass

try:
    p = metrics_dir / "cnn.json"
    if p.exists():
        cnn_meta = json.loads(p.read_text())
except Exception:
    pass

champion_obj = globals().get("champion_obj", None)
status = champion_obj.get("status") if isinstance(champion_obj, dict) else None

def _fmt(x, nd=3):
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x) if x is not None else "n/a"

# Build rows
rows = []
if best_tree:
    rows.append(["best_tree", best_tree.get("best_tree_name","tree"),
                 _fmt(best_tree.get("best_tree_val_acc")), "metrics/best_tree.json"])
if cnn_meta:
    rows.append(["cnn", "cnn_benchmark",
                 _fmt(cnn_meta.get("val_acc")), "metrics/cnn.json"])

winner = None
if champion_obj and status == "selected":
    winner = champion_obj.get("winner_name","?")
    rows.append(["champion", champion_obj.get("winner_name","?"),
                 _fmt(champion_obj.get("val_acc", champion_obj.get("val_ap"))),
                 f"models/{champion_obj.get('model_file','(n/a)')}"])
elif champion_obj:
    rows.append(["champion", f"({champion_obj.get('status','pending')})",
                 "n/a", "reports/champion.json"])

# Pretty print (no extra deps)
colw = [12, 24, 12, 36]
headers = ["source", "name", "metric", "path"]
print(" ".join(h.ljust(w) for h, w in zip(headers, colw)))
print("-" * sum(colw))
for r in rows:
    r = [str(x) for x in r]
    # truncate long values conservatively
    r[3] = (r[3][:colw[3]-3] + "...") if len(r[3]) > colw[3] else r[3]
    print(" ".join(v.ljust(w) for v, w in zip(r, colw)))

print(f"[status] champion: {status or 'n/a'}")
print(">>> Section 9.8 — Champion Summary: complete")